# 03 — Label Pipeline
Compute labels from the held-out label window, with an explicit leakage check.

In [1]:
import sys
sys.path.insert(0, "..")
from _lib import load_search_data, assert_no_window_overlap, compute_labels, FEATURE_WEEKS, LABEL_WEEKS, TEST_LABEL_WEEKS
import pandas as pd

df = load_search_data()
assert_no_window_overlap()

Leakage check passed: feature / label / test windows are disjoint.
  feature weeks: 0-11
  label weeks (train/val target): 12-15
  test weeks (held out): 16-19


In [2]:
labels = compute_labels(df, LABEL_WEEKS)
labels['label'].value_counts(normalize=True).round(3)

label
stable       0.632
declining    0.200
growing      0.168
Name: proportion, dtype: float64

In [3]:
# Same logic applied to the test window, for final held-out evaluation later
test_labels = compute_labels(df, TEST_LABEL_WEEKS)
test_labels['label'].value_counts(normalize=True).round(3)

label
stable       0.608
declining    0.214
growing      0.178
Name: proportion, dtype: float64

In [4]:
labels.to_parquet(f"{'..'}/work/data_cache_labels_train.parquet") if False else None
import os
os.makedirs("data_cache", exist_ok=True)
labels.to_parquet("data_cache/labels_train.parquet")
test_labels.to_parquet("data_cache/labels_test.parquet")
print("Saved label sets to work/data_cache/")

Saved label sets to work/data_cache/


## Sanity check
Label distribution is imbalanced toward `stable`, which matches expectation —
most pages don't move dramatically week to week. `growing`/`declining` are the
minority classes we actually care about surfacing for action.